<a href="https://colab.research.google.com/github/neelameghana192311002/CSA6301---Threat-Intelligence-and-Network-Security/blob/main/UNIT4LAB/Exercise_1_Packet_Filtering_Firewall_Rule_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
def rule_matches(packet, rule):
    def field_ok(value, rule_value):
        return rule_value == "any" or value == rule_value

    return (
        field_ok(packet["src"], rule["src"])
        and field_ok(packet["dst"], rule["dst"])
        and field_ok(packet["port"], rule["port"])
        and field_ok(packet["proto"], rule["proto"])
    )


def evaluate_packet(packet, rules):
    """
    Return the action of the first matching rule (top-down),
    or 'deny' if no rule matches (default-deny).
    """
    for rule in rules:
        if rule_matches(packet, rule):
            return rule["action"]

    return "deny"

In [3]:
def test_experiment1():
    # Misconfigured order: a broad allow placed before a specific deny
    misordered_rules = [
        {"action": "allow", "src": "any", "dst": "any", "port": "any", "proto": "any"},
        {"action": "deny", "src": "10.0.0.5", "dst": "any", "port": "any", "proto": "any"},
    ]

    packet = {
        "src": "10.0.0.5",
        "dst": "8.8.8.8",
        "port": 443,
        "proto": "tcp"
    }

    # BUG: Rule 1 matches everything and comes first,
    # so the deny never triggers
    assert evaluate_packet(packet, misordered_rules) == "allow"

    # Fixed order: specific deny placed before the broad allow
    fixed_rules = [
        {"action": "deny", "src": "10.0.0.5", "dst": "any", "port": "any", "proto": "any"},
        {"action": "allow", "src": "any", "dst": "any", "port": "any", "proto": "any"},
    ]

    assert evaluate_packet(packet, fixed_rules) == "deny"

    # Default-deny: no rule matches an unrelated packet
    other_packet = {
        "src": "192.168.1.9",
        "dst": "1.1.1.1",
        "port": 53,
        "proto": "udp"
    }

    assert evaluate_packet(other_packet, []) == "deny"

    print("All test cases passed.")


test_experiment1()

All test cases passed.
